# PCA and Cluster Analysis of Financial Ratios
### 100 Indian listed companies · 4 industries · FY2017–FY2026

This notebook answers two different questions, in order:

1. **Grouping the ratios (PCA).** Many of the 25 ratios measure the same thing (ROA, ROE and net margin all measure profitability). Principal Component Analysis reduces them to a few underlying financial dimensions and shows which ratios belong together — following Pinches, Mingo & Caruthers (1973) and Chen & Shimerda (1981).
2. **Grouping the companies (cluster analysis).** Companies are grouped by how similar their financial profiles are, and the groups are compared with their industries — following Gupta & Huefner (1972).

**How to run:** open this file in VS Code, select your Python environment as the kernel (top right), and click **Run All**. Your data file must be at `data/financials.xlsx`.

## 0 · Setup

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import kruskal
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

# ---- Settings (change here if needed) ----
DATA_FILE = Path(os.environ.get("FINANCIALS_PATH", "data/financials.xlsx"))  # your workbook
OUTPUT_DIR = Path(os.environ.get("OUTPUT_DIR", "output"))  # CSV exports for jamovi
WINSOR = 0.05  # cap each ratio at its 5th and 95th percentile
K = 4  # number of company clusters (4 = compare with the 4 industries)
RANDOM_STATE = 0  # makes k-means give the same result every run

# Chart colours
BLUE, GREY, RED = "#2a78d6", "#a3a29d", "#e34948"
CLUSTER_COLOURS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
MARKERS = ["o", "s", "^", "D", "P", "v"]

## 1 · Load the data

The workbook has one row per company per fiscal year (`Data` sheet). Row 2 holds the column names and row 3 the Capital IQ codes, which are skipped.
Screener does not report some items for some companies (e.g. inventory at IT firms); those cells were filled with 0 and noted in the `Notes` column. They are turned back into **missing** values here, because "not reported" is not the same as zero.

In [ ]:
df = pd.read_excel(DATA_FILE, sheet_name="Data", header=1)
if str(df.iloc[0]["Total Revenue"]).startswith("IQ_"):  # Capital IQ code row
    df = df.iloc[1:]
df = df.dropna(subset=["Company"]).reset_index(drop=True)

ITEMS = [
    "Total Revenue", "Cost of Goods Sold", "Gross Profit", "EBIT", "EBITDA", "Interest Expense", "Net Income",
    "Total Current Assets", "Total Current Liabilities", "Cash & ST Investments", "Inventory", "Total Receivables",
    "Total Assets", "Total Debt", "Total Equity", "Cash from Operations", "Debt Repaid", "Dividend per Share",
    "Share Price (period end)", "Market Cap", "Total Enterprise Value",
]
for col in ITEMS:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Fiscal year label, e.g. "FY2026"
df["Year"] = df["Period"].astype(str)
no_label = ~df["Year"].str.fullmatch(r"FY\d{4}")
df.loc[no_label, "Year"] = "FY" + pd.to_datetime(df.loc[no_label, "Period End Date"]).dt.year.astype(str)

# "Not reported on Screener, set to 0: inventory, debt repaid"  ->  missing
n_fixed = 0
for i, note in df.get("Notes", pd.Series(dtype=str)).dropna().astype(str).items():
    if "set to 0:" in note:
        for item in note.split("set to 0:")[1].split(";")[0].split(","):
            col = next((c for c in ITEMS if c.lower() == item.strip().lower()), None)
            if col:
                df.at[i, col] = np.nan
                n_fixed += 1

print(f"{df['Company'].nunique()} companies, {df['Year'].nunique()} years ({df['Year'].min()}–{df['Year'].max()}), {len(df)} rows")
print(f"{n_fixed} 'not reported' values treated as missing")
print(df.groupby("Industry")["Company"].nunique())
df[["Industry", "Company", "Year"] + ITEMS[:6]].head()

## 2 · Calculate the 25 ratios

Each ratio is calculated for every company and year. A ratio is left **blank** when it would be meaningless — for example P/E when the company made a loss, ROE when equity is negative, or inventory turnover when no inventory is reported. Blank values are handled in Step 3.

In [ ]:
def div(a, b):
    # a / b, blank when b is missing, zero or negative (ratio not meaningful)
    return a / b.where(b > 0)

r = df[["Industry", "Company", "Year"]].copy()
cl, rev = df["Total Current Liabilities"], df["Total Revenue"]

# Liquidity
r["Current Ratio"] = div(df["Total Current Assets"], cl)
r["Quick Ratio"] = div(df["Total Current Assets"] - df["Inventory"].fillna(0), cl)
r["Cash Ratio"] = div(df["Cash & ST Investments"], cl)
r["Operating Cash Flow Ratio"] = div(df["Cash from Operations"], cl)
r["Working Capital Ratio"] = div(df["Total Current Assets"] - cl, df["Total Assets"])
# Profitability (%)
r["Gross Profit Margin"] = 100 * div(df["Gross Profit"], rev)
r["Operating Profit Margin"] = 100 * div(df["EBIT"], rev)
r["Net Profit Margin"] = 100 * div(df["Net Income"], rev)
r["Return on Assets (ROA)"] = 100 * div(df["Net Income"], df["Total Assets"])
r["Return on Equity (ROE)"] = 100 * div(df["Net Income"], df["Total Equity"])
# Efficiency
r["Asset Turnover"] = div(rev, df["Total Assets"])
r["Inventory Turnover"] = div(df["Cost of Goods Sold"], df["Inventory"])
r["Receivables Turnover"] = div(rev, df["Total Receivables"])
r["Days Sales Outstanding (DSO)"] = div(365 + 0 * rev, r["Receivables Turnover"])
r["Days Inventory Outstanding (DIO)"] = div(365 + 0 * rev, r["Inventory Turnover"])
# Leverage
r["Debt-to-Equity"] = div(df["Total Debt"], df["Total Equity"])
r["Debt-to-Assets"] = div(df["Total Debt"], df["Total Assets"])
r["Interest Coverage Ratio"] = div(df["EBIT"], df["Interest Expense"])
r["Equity Multiplier"] = div(df["Total Assets"], df["Total Equity"])
r["Debt Service Coverage Ratio (DSCR)"] = div(df["EBITDA"], df["Interest Expense"] + df["Debt Repaid"].fillna(0))
# Valuation
r["Price-to-Earnings (P/E)"] = div(df["Market Cap"], df["Net Income"])
r["Price-to-Sales (P/S)"] = div(df["Market Cap"], rev)
r["Price-to-Book (P/B)"] = div(df["Market Cap"], df["Total Equity"])
r["EV/EBITDA"] = div(df["Total Enterprise Value"], df["EBITDA"])
r["Dividend Yield"] = 100 * div(df["Dividend per Share"], df["Share Price (period end)"])

CATEGORY = {
    "Liquidity": ["Current Ratio", "Quick Ratio", "Cash Ratio", "Operating Cash Flow Ratio", "Working Capital Ratio"],
    "Profitability": ["Gross Profit Margin", "Operating Profit Margin", "Net Profit Margin", "Return on Assets (ROA)", "Return on Equity (ROE)"],
    "Efficiency": ["Asset Turnover", "Inventory Turnover", "Receivables Turnover", "Days Sales Outstanding (DSO)", "Days Inventory Outstanding (DIO)"],
    "Leverage": ["Debt-to-Equity", "Debt-to-Assets", "Interest Coverage Ratio", "Equity Multiplier", "Debt Service Coverage Ratio (DSCR)"],
    "Valuation": ["Price-to-Earnings (P/E)", "Price-to-Sales (P/S)", "Price-to-Book (P/B)", "EV/EBITDA", "Dividend Yield"],
}
RATIOS = [x for group in CATEGORY.values() for x in group]
TEXTBOOK = {ratio: cat for cat, group in CATEGORY.items() for ratio in group}
r[RATIOS] = r[RATIOS].replace([np.inf, -np.inf], np.nan)

print(f"{len(RATIOS)} ratios for {len(r)} company-years")
print("Blank (not meaningful) values per ratio:")
print(r[RATIOS].isna().sum().loc[lambda s: s > 0].sort_values(ascending=False).to_string())
r.head()

## 3 · Prepare the data points for PCA and clustering

| Step | What | Why |
|---|---|---|
| a | **One data point = one company**: the median of each ratio over FY2017–FY2026 | PCA and clustering need one row per company; the median is not distorted by one unusual year |
| b | Leave out 4 ratios | Inventory Turnover and DIO cannot be calculated for most IT companies; Receivables Turnover duplicates DSO (DSO = 365 / turnover) and Equity Multiplier duplicates Debt-to-Equity, so keeping both would count the same information twice |
| c | Fill the few remaining gaps with the median of the company's own industry | PCA needs a complete table |
| d | Cap each ratio at its 5th and 95th percentile (winsorising) | Financial ratios are highly skewed; without this a few extreme companies dominate (Lev & Sunder, 1979) |
| e | Standardise each ratio (z-score: mean 0, standard deviation 1) | Ratios in %, x and days then count equally |

In [ ]:
EXCLUDE = ["Inventory Turnover", "Days Inventory Outstanding (DIO)", "Receivables Turnover", "Equity Multiplier"]
USED = [x for x in RATIOS if x not in EXCLUDE]

# a) one row per company: median over all years
profiles = r.groupby("Company", sort=False)[RATIOS].median()
industry = r.groupby("Company", sort=False)["Industry"].first()

# b) chosen ratios
X = profiles[USED].copy()

# c) fill gaps with the industry median
print("Missing values filled:", X.isna().sum()[lambda s: s > 0].to_dict())
X = X.fillna(X.groupby(industry).transform("median")).fillna(X.median())

# d) cap extreme values
low, high = X.quantile(WINSOR), X.quantile(1 - WINSOR)
print(f"Values capped at the {WINSOR:.0%}/{1 - WINSOR:.0%} percentiles:", int(((X < low) | (X > high)).sum().sum()))
X = X.clip(low, high, axis=1)

# e) standardise
Z = (X - X.mean()) / X.std(ddof=1)

print(f"\nData points: {Z.shape[0]} companies × {Z.shape[1]} ratios")
X.round(2).head()

## 4 · Step 1 — Grouping the ratios with PCA

PCA is run on the **correlation matrix** of the standardised ratios.
- Keep components with an **eigenvalue above 1** — each must explain more than one ratio's worth of variance (Kaiser, 1960).
- Rotate them with **varimax** so each ratio loads mainly on one component, which makes them easier to name (Kaiser, 1958).
These are also jamovi's and SPSS's defaults, so the results can be checked there.

In [ ]:
corr = np.corrcoef(Z.values, rowvar=False)
eigenvalues, eigenvectors = np.linalg.eigh(corr)
order = np.argsort(eigenvalues)[::-1]
eigenvalues, eigenvectors = eigenvalues[order], eigenvectors[:, order]

n_comp = int((eigenvalues > 1).sum())
explained = pd.DataFrame({
    "Eigenvalue": eigenvalues,
    "% of variance": 100 * eigenvalues / eigenvalues.sum(),
}, index=[f"PC{i + 1}" for i in range(len(eigenvalues))])
explained["Cumulative %"] = explained["% of variance"].cumsum()
print(f"Components kept (eigenvalue > 1): {n_comp}, explaining {explained['Cumulative %'].iloc[n_comp - 1]:.1f}% of the variance")

# Scree plot
fig, ax = plt.subplots(figsize=(9, 4))
shown = explained.head(12)
ax.bar(shown.index, shown["Eigenvalue"], color=[BLUE if i < n_comp else GREY for i in range(len(shown))])
ax.axhline(1, color=RED, linestyle="--", linewidth=1, label="Eigenvalue = 1")
ax.set_ylabel("Eigenvalue"); ax.set_title("Scree plot (blue = components kept)"); ax.legend()
ax.spines[["top", "right"]].set_visible(False)
plt.show()

explained.head(n_comp).round(2)

In [ ]:
def varimax(L, max_iter=500, tol=1e-8):
    # Varimax rotation with Kaiser normalisation (Kaiser, 1958)
    p, k = L.shape
    h = np.sqrt((L ** 2).sum(axis=1, keepdims=True))
    A = L / h
    R, d_old = np.eye(k), 0
    for _ in range(max_iter):
        B = A @ R
        u, s, vt = np.linalg.svd(A.T @ (B ** 3 - B @ np.diag((B ** 2).sum(axis=0)) / p))
        R = u @ vt
        if d_old and s.sum() / d_old < 1 + tol:
            break
        d_old = s.sum()
    return (A @ R) * h

# Unrotated loadings = eigenvector × √eigenvalue, then rotate
loadings = varimax(eigenvectors[:, :n_comp] * np.sqrt(eigenvalues[:n_comp]))
order = np.argsort((loadings ** 2).sum(axis=0))[::-1]  # biggest component first
loadings = loadings[:, order]
loadings *= np.where(np.abs(loadings.max(axis=0)) >= np.abs(loadings.min(axis=0)), 1, -1)  # largest loading positive
COMPS = [f"PC{i + 1}" for i in range(n_comp)]
loadings = pd.DataFrame(loadings, index=USED, columns=COMPS)

# Name each component after the textbook category of its strongest ratio
top = loadings.abs().idxmax()
cats = top.map(TEXTBOOK)
NAMES = {c: cats[c] if (cats == cats[c]).sum() == 1 else f"{cats[c]} ({top[c]})" for c in COMPS}
for c in COMPS:
    print(f"{c}: {NAMES[c]}")

In [ ]:
# Which component does each ratio belong to? (= the grouping of the ratios)
main = loadings.abs().idxmax(axis=1)
groups = pd.DataFrame({
    "Component": main.map(lambda c: f"{c}: {NAMES[c]}"),
    "Loading": [loadings.at[x, c] for x, c in main.items()],
    "Textbook category": [TEXTBOOK[x] for x in USED],
    "Communality": (loadings ** 2).sum(axis=1),
}, index=USED)
groups["Matches textbook?"] = ["✓" if t in comp else "✗" for t, comp in zip(groups["Textbook category"], groups["Component"])]
groups = groups.sort_values(["Component", "Loading"], ascending=[True, False])

table = loadings.loc[groups.index].map(lambda v: f"{v:+.2f}" if abs(v) >= 0.4 else "")  # hide small loadings
table.columns = [f"{c}: {NAMES[c]}" for c in COMPS]
table["Textbook category"] = groups["Textbook category"]
table["Matches textbook?"] = groups["Matches textbook?"]
print(f"{(groups['Matches textbook?'] == '✓').sum()} of {len(groups)} ratios group where the textbook puts them.")
print("Exceptions:", ", ".join(groups.index[groups["Matches textbook?"] == "✗"]))
table

**Reading the table:** each ratio is listed under the component it loads on most (loadings below 0.40 hidden). A loading near ±1 means the ratio is almost entirely that component. *Communality* (in `groups`) is the share of the ratio's variance the kept components explain.

For comparison: Pinches et al. (1973) found about 7 stable ratio factors and Chen & Shimerda (1981) 7; Salmi, Virtanen & Yli-Olli (1990) found 6 and that the textbook categories were not directly supported. Gombola & Ketz (1983) found cash-flow ratios can form a factor of their own.

## 5 · Step 2 — Grouping the companies with cluster analysis

**Data points:** the 100 companies. **What each company is described by:** its component scores from Step 1 (not the raw ratios), so overlapping ratios such as ROA, ROE and net margin do not count profitability several times.

**Method: k-means** (MacQueen, 1967)
1. Choose the number of clusters *k*.
2. Place *k* starting centres among the companies' scores.
3. Assign every company to its **nearest centre** (straight-line distance across all component scores).
4. Move each centre to the **average** of its companies.
5. Repeat 3–4 until no company changes cluster.
6. Do this from 50 random starts and keep the solution with the smallest total distance. Clusters are numbered by size (1 = largest).

**Choosing k:** the average **silhouette** (Rousseeuw, 1987) measures how much closer companies are to their own cluster than to the next one (above 0.5 strong, 0.25–0.5 moderate, below 0.25 weak).
**Industry is not used to form the clusters** — it is only used afterwards to compare, as in Gupta & Huefner (1972).

In [ ]:
# Component scores for each company (regression method), standardised
weights = np.linalg.solve(corr, loadings.values)
scores = pd.DataFrame(Z.values @ weights, index=Z.index, columns=COMPS)
scores = (scores - scores.mean()) / scores.std(ddof=1)

# Silhouette for k = 2..8
sil = {}
for k in range(2, 9):
    lab = KMeans(k, n_init=50, random_state=RANDOM_STATE).fit_predict(scores)
    sil[k] = silhouette_score(scores, lab)
sil = pd.Series(sil, name="Average silhouette")

fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(sil.index.astype(str), sil.values, color=[BLUE if k == K else GREY for k in sil.index])
ax.set_xlabel("Number of clusters (k)"); ax.set_ylabel("Silhouette")
ax.set_title(f"Silhouette by k (blue = chosen k = {K}; best = {sil.idxmax()})")
ax.spines[["top", "right"]].set_visible(False)
plt.show()
sil.round(3).to_frame().T

In [ ]:
km = KMeans(K, n_init=50, random_state=RANDOM_STATE).fit(scores)

# Number clusters by size: 1 = largest
size_order = pd.Series(km.labels_).value_counts().index
relabel = {old: new + 1 for new, old in enumerate(size_order)}
cluster = pd.Series([relabel[x] for x in km.labels_], index=scores.index, name="Cluster")
centres = pd.DataFrame(km.cluster_centers_, columns=COMPS, index=[relabel[i] for i in range(K)]).sort_index()
centres.index.name = "Cluster"
distance = pd.Series(np.linalg.norm(scores.values - centres.loc[cluster].values, axis=1), index=scores.index)

# Checks: agreement with industries and with a second method (Ward, 1963), by Adjusted Rand Index (Hubert & Arabie, 1985)
ward = AgglomerativeClustering(n_clusters=K, linkage="ward").fit_predict(scores)
print(f"k = {K}, silhouette = {sil[K]:.2f}")
print(f"Match with industries (ARI, 0 = chance, 1 = identical): {adjusted_rand_score(industry[cluster.index], cluster):.2f}")
print(f"Agreement with Ward clustering (ARI): {adjusted_rand_score(cluster, ward):.2f}")

print("\nCluster centres (average component scores; 0 = all-company average):")
centres.rename(columns=lambda c: f"{c}: {NAMES[c]}").round(2)

In [ ]:
# Companies on the first two components: colour = cluster, shape = industry
fig, ax = plt.subplots(figsize=(10, 7))
for j, ind in enumerate(industry.unique()):
    for c in range(1, K + 1):
        sel = (industry[scores.index] == ind) & (cluster == c)
        ax.scatter(scores.loc[sel, COMPS[0]], scores.loc[sel, COMPS[1]], c=CLUSTER_COLOURS[c - 1],
                   marker=MARKERS[j % len(MARKERS)], s=60, edgecolors="white", linewidths=0.8)
ax.set_xlabel(f"{COMPS[0]}: {NAMES[COMPS[0]]}"); ax.set_ylabel(f"{COMPS[1]}: {NAMES[COMPS[1]]}")
ax.set_title("Companies on the first two components")
from matplotlib.lines import Line2D
handles = [Line2D([], [], marker="o", ls="", color=CLUSTER_COLOURS[c - 1], label=f"Cluster {c}") for c in range(1, K + 1)]
handles += [Line2D([], [], marker=MARKERS[j % len(MARKERS)], ls="", color="#52514e", label=ind) for j, ind in enumerate(industry.unique())]
ax.legend(handles=handles, loc="center left", bbox_to_anchor=(1, 0.5), frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

print("Clusters vs industries (number of companies):")
pd.crosstab(cluster.map(lambda c: f"Cluster {c}"), industry[cluster.index], margins=True, margins_name="Total")

In [ ]:
# What characterises each cluster: median ratio values per cluster
cluster_profiles = profiles[USED].groupby(cluster).median().T
cluster_profiles.columns = [f"Cluster {c}" for c in cluster_profiles.columns]
cluster_profiles["All companies"] = profiles[USED].median()

zmeans = Z.groupby(cluster).mean()
for c in zmeans.index:
    row = zmeans.loc[c].sort_values()
    high = [f"high {x}" for x in row.index[::-1][:3] if row[x] > 0.3]
    low = [f"low {x}" for x in row.index[:2] if row[x] < -0.3]
    print(f"Cluster {c} ({(cluster == c).sum()} companies): " + "; ".join(high + low))

cluster_profiles.loc[groups.index].round(2)

In [ ]:
# The data points: each company's component scores, cluster and distance to its centre
points = scores.round(2).rename(columns=lambda c: f"{c}: {NAMES[c]}")
points.insert(0, "Industry", industry[points.index])
points["Cluster"] = cluster
points["Distance to centre"] = distance.round(2)
for c in range(1, K + 1):
    members = points.index[points["Cluster"] == c]
    print(f"Cluster {c}: " + ", ".join(f"{m} ({industry[m].split()[0]})" for m in members) + "\n")
points.sort_values(["Cluster", "Distance to centre"])

## 6 · Step 3 — Which ratios really separate the industries?

A **Kruskal–Wallis test** (Kruskal & Wallis, 1952) on each ratio's company medians across the industries — a rank-based test, so skewed ratios are fine. The effect size **eta² = (H − k + 1) / (n − k)** is the share of the ratio's variation explained by industry (Tomczak & Tomczak, 2014): 0.01 small, 0.06 medium, 0.14+ large.

In [ ]:
rows = []
for x in USED:
    samples = [profiles.loc[industry == g, x].dropna() for g in industry.unique()]
    H, p = kruskal(*samples)
    n, k = sum(len(s) for s in samples), len(samples)
    med = profiles[x].groupby(industry).median()
    rows.append({"Ratio": x, "H": H, "p-value": p, "eta²": max((H - k + 1) / (n - k), 0),
                 "Highest industry": med.idxmax(), "Lowest industry": med.idxmin()})
separation = pd.DataFrame(rows).sort_values("eta²", ascending=False).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(8, 7))
s = separation.iloc[::-1]
ax.barh(s["Ratio"], s["eta²"], color=[BLUE if p < 0.05 else GREY for p in s["p-value"]])
ax.set_xlabel("eta² (variation explained by industry)"); ax.set_title("Blue = significant difference (p < 0.05)")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()
separation.round(4)

## 7 · Export for jamovi

Saves `output/pca_cluster_data.csv` (ratios used, component scores and cluster per company) and `output/ratios_by_company_year.csv` (all 25 ratios). The `output/` folder is not uploaded to GitHub.

To reproduce in jamovi:
1. Open `pca_cluster_data.csv`.
2. **Factor → Principal Component Analysis** on the ratio columns (not the PC or Cluster columns), *eigenvalue > 1*, *Varimax* — loadings match Step 1 (signs may flip).
3. Install the **snowCluster** module and run k-means on the PC columns with the same *k*.
4. **ANOVA → One-Way ANOVA (Non-parametric)** on each ratio by Industry reproduces Step 3.

In [ ]:
OUTPUT_DIR.mkdir(exist_ok=True)
export = pd.concat([industry.rename("Industry"), X.round(4),
                    scores.round(4).rename(columns=lambda c: f"{c} {NAMES[c]}"), cluster], axis=1)
export.rename_axis("Company").reset_index().to_csv(OUTPUT_DIR / "pca_cluster_data.csv", index=False)
r.round(4).to_csv(OUTPUT_DIR / "ratios_by_company_year.csv", index=False)
print("Saved:", *sorted(p.name for p in OUTPUT_DIR.glob("*.csv")))

## References

**Financial ratio studies**
- Chen, K. H., & Shimerda, T. A. (1981). An empirical analysis of useful financial ratios. *Financial Management, 10*(1), 51–60.
- Gombola, M. J., & Ketz, J. E. (1983). A note on cash flow and classification patterns of financial ratios. *The Accounting Review, 58*(1), 105–114.
- Gupta, M. C., & Huefner, R. J. (1972). A cluster analysis study of financial ratios and industry characteristics. *Journal of Accounting Research, 10*(1), 77–95.
- Lev, B., & Sunder, S. (1979). Methodological issues in the use of financial ratios. *Journal of Accounting and Economics, 1*(3), 187–210.
- Pinches, G. E., Mingo, K. A., & Caruthers, J. K. (1973). The stability of financial patterns in industrial organizations. *Journal of Accounting Research, 11*(2), 389–396.
- Salmi, T., Virtanen, I., & Yli-Olli, P. (1990). *On the classification of financial ratios: A factor and transformation analysis of accrual, cash flow, and market-based ratios.* Acta Wasaensia No. 25, University of Vaasa.

**Statistical methods**
- Hubert, L., & Arabie, P. (1985). Comparing partitions. *Journal of Classification, 2*(1), 193–218.
- Kaiser, H. F. (1958). The varimax criterion for analytic rotation in factor analysis. *Psychometrika, 23*(3), 187–200.
- Kaiser, H. F. (1960). The application of electronic computers to factor analysis. *Educational and Psychological Measurement, 20*(1), 141–151.
- Kruskal, W. H., & Wallis, W. A. (1952). Use of ranks in one-criterion variance analysis. *Journal of the American Statistical Association, 47*(260), 583–621.
- MacQueen, J. (1967). Some methods for classification and analysis of multivariate observations. *Proceedings of the Fifth Berkeley Symposium on Mathematical Statistics and Probability, 1*, 281–297.
- Rousseeuw, P. J. (1987). Silhouettes: A graphical aid to the interpretation and validation of cluster analysis. *Journal of Computational and Applied Mathematics, 20*, 53–65.
- Tomczak, M., & Tomczak, E. (2014). The need to report effect size estimates revisited: An overview of some recommended measures of effect size. *Trends in Sport Sciences, 21*(1), 19–25.
- Ward, J. H. (1963). Hierarchical grouping to optimize an objective function. *Journal of the American Statistical Association, 58*(301), 236–244.

**Data:** company financial statements and share prices from Screener.in (consolidated accounts, FY2017–FY2026).